# Crawl One Website (Bare Minimum Crawler)

In [1]:
import requests

url = 'https://cnn.com'
html = requests.get(url).text

In [2]:
html[0:500]

'  <!DOCTYPE html>\n<html lang="en" data-uri="cms.cnn.com/_pages/clg34ol9u000047nodabud1o2@published" data-layout-uri="cms.cnn.com/_layouts/layout-homepage/instances/homepage-domestic@published"  data-site="cnn">\n  <head>\n<link rel="preload" href="/fonts/cnn/cnn_sans_display-v1.woff2" as="font" type="font/woff2" crossorigin="anonymous">\n<link rel="preload" href="/fonts/cnn/cnn_sans_display-medium-v1.woff2" as="font" type="font/woff2" crossorigin="anonymous">\n<link rel="preload" href="/fonts/cnn/cn'

# Crawl One Website (Extended)

In [3]:
import requests
from bs4 import BeautifulSoup

In [4]:
url = 'https://cnn.com'

html = requests.get(url).text
soup = BeautifulSoup(html, "html.parser")

In [5]:
links = [a.get("href") for a in soup.find_all("a")]
links[0:10]

['https://www.cnn.com',
 'https://www.cnn.com/us',
 'https://www.cnn.com/world',
 'https://www.cnn.com/politics',
 'https://www.cnn.com/business',
 'https://www.cnn.com/health',
 'https://www.cnn.com/entertainment',
 'https://www.cnn.com/cnn-underscored',
 'https://www.cnn.com/style',
 'https://www.cnn.com/travel']

In [6]:
images = sorted(set([img.get("src") for img in soup.find_all("img") if img.get("src") is not None]))
images[0:10]

['/media/sites/cnn/app-store-cnn-app-qr-code.png',
 '/media/sites/cnn/google-play-cnn-app-qr-code.png',
 'https://media.cnn.com/api/v1/images/stellar/bleacherreport/20260811133915064-getty-new-york-derek-jeter-of-the-new-york-yankees-runs-back-to-the-dugout-against-the-texas-rangers.png?c=2x3&q=h_384,w_256,c_fill',
 'https://media.cnn.com/api/v1/images/stellar/bleacherreport/2268220349-0-large-cropped.jpg?c=2x3&q=h_384,w_256,c_fill',
 'https://media.cnn.com/api/v1/images/stellar/bleacherreport/2286082000-large-cropped.jpg?c=2x3&q=h_384,w_256,c_fill',
 'https://media.cnn.com/api/v1/images/stellar/bleacherreport/ezgif-586e6a6d94304ea3-1.gif?c=16x9&q=h_438,w_780,c_fill',
 'https://media.cnn.com/api/v1/images/stellar/bleacherreport/jets-jaguars-football-76589-7721x4344-0-140.jpg?c=2x3&q=h_384,w_256,c_fill',
 'https://media.cnn.com/api/v1/images/stellar/prod/01-rain-totals-observed-081126.png?c=2x3&q=h_384,w_256,c_fill',
 'https://media.cnn.com/api/v1/images/stellar/prod/08-c02-202608041503

# Pull Full URL Context

In [7]:
import re, json, requests, trafilatura, nltk
import pandas as pd

from collections import Counter
from nltk.tokenize import sent_tokenize
from scipy.stats import entropy

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

from tqdm import tqdm

# add/run these to prevent the nltk silent killer

nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\itsgo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\itsgo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [8]:
def extract_web_text_context(url):

    response = requests.get(url, timeout=5)

    result = trafilatura.extract(response.text, output_format="json")

    if result is None:
        return None

    data = json.loads(result)

    domain = urlparse(url).netloc.replace("www.", "")

    soup = BeautifulSoup(response.text, "html.parser")

    page_title = None

    if soup.title and soup.title.string:
        page_title = soup.title.string.strip()

    data['page_title'] = page_title

    links = []

    for anchor in soup.find_all("a", href=True):
        links.append(urljoin(url, anchor["href"]))

    for iframe in soup.find_all("iframe", src=True):
        links.append(urljoin(url, iframe["src"]))

    links = [link.replace("www.", "") for link in links]
    links = sorted(set(links))

    images = []

    for img in soup.find_all("img", src=True):
        images.append(urljoin(url, img["src"]))

    images = sorted(set(images))

    linked_domains = [urlparse(link).netloc for link in links]
    linked_domains = sorted(set(linked_domains))

    data['links'] = links
    data['images'] = images
    data['domain'] = domain
    data['linked_domains'] = linked_domains

    text = data['text']

    text = text.replace("\n", " ").replace("\t", " ")
    text = re.sub(r" +", " ", text)
    text = text.strip()

    data['url'] = url
    data['text'] = text
    data['tokens'] = text.split()

    sentences = sent_tokenize(text)
    data['sentences'] = sentences

    # summary statistics
    data['token_count'] = len(data['tokens'])
    data['sentence_count'] = len(data['sentences'])
    data['lexical_diversity'] = len(set(data['tokens'])) / data['token_count']

    # Shannon's Entropy - Information Theory 
    counts = Counter(data['tokens'])
    data['entropy'] = float(entropy(list(counts.values()), base=2))

    return data

In [9]:
url = 'https://cnn.com'

data = extract_web_text_context(url)
data.keys()

dict_keys(['text', 'comments', 'page_title', 'links', 'images', 'domain', 'linked_domains', 'url', 'tokens', 'sentences', 'token_count', 'sentence_count', 'lexical_diversity', 'entropy'])

In [10]:
data['text'][0:1000]

'- • For Subscribers• For SubscribersFor Subscribers ‘Fireworks suck’: Removed El-Sayed videos urged getting rid of the 4th of July tradition, raised replacing 2nd Amendment - What to watch in Tuesday’s primaries in Wisconsin, Minnesota, South Carolina and more - • Streaming NowStreaming NowStreaming Now MyPillow founder. Election denier. Former crack addict. Governor? - • For SubscribersFor Subscribers Have questions about today’s primaries? Ask CNN’s John King - He flew out of Turkey on a smaller craft; reporters and staffers were unknowingly on a decoy plane - • Video 0:22Video 0:22Video Rubio ignores question on if administration put Americans at risk on Air Force One 0:22 - • AnalysisAnalysis Trump’s secret plane swap is more problematic than it might seem Data centers - • For Subscribers• For SubscribersFor Subscribers ‘Brand-new everything’: How data centers transformed this small town’s economy - • Streaming NowStreaming NowStreaming Now The rise of AI brings fear over the huge

In [11]:
data['page_title']

'Breaking News, Latest News and Videos | CNN'

In [12]:
data['links']

['https://arabic.cnn.com?hpt=header_edition-picker',
 'https://bleacherreport.com/',
 'https://bleacherreport.com/?utm_source=cnn.com&utm_medium=referral&utm_campaign=editorial',
 'https://bleacherreport.com/articles/25469858-ranking-10-best-sports-uniforms-all-time?utm_source=cnn.com&utm_medium=referral&utm_campaign=editorial',
 'https://bleacherreport.com/articles/25470082-golfer-who-faked-hole-one-during-tournament-resigns-club-after-viral-video?utm_source=cnn.com&utm_medium=referral&utm_campaign=editorial',
 'https://bleacherreport.com/articles/25470215-christmas-2026-features-loaded-nfl-and-nba-schedules-heres-everything-you-need-know?utm_source=cnn.com&utm_medium=referral&utm_campaign=editorial',
 'https://bleacherreport.com/articles/25470219-jets-qwantez-stiggers-hospitalized-after-leaving-practice-ambulance?utm_source=cnn.com&utm_medium=referral&utm_campaign=editorial',
 'https://bleacherreport.com/articles/25470225-new-aaron-donald-workout-video-drops-amid-nfl-comeback-rumors-

In [13]:
data['domain']

'cnn.com'

In [14]:
data['linked_domains']

['arabic.cnn.com',
 'bleacherreport.com',
 'careers.wbd.com',
 'cnn.com',
 'cnn.it',
 'cnn.onelink.me',
 'cnn10.com',
 'cnnespanol.cnn.com',
 'edition.cnn.com',
 'facebook.com',
 'help.cnn.com',
 'instagram.com',
 'linkedin.com',
 'threads.com',
 'tiktok.com',
 'twitter.com',
 'us.cnn.com']

In [15]:
data['url']

'https://cnn.com'

In [16]:
data['tokens'][0:10]

['-',
 '•',
 'For',
 'Subscribers•',
 'For',
 'SubscribersFor',
 'Subscribers',
 '‘Fireworks',
 'suck’:',
 'Removed']

In [17]:
data['sentences'][0:10]

['- • For Subscribers• For SubscribersFor Subscribers ‘Fireworks suck’: Removed El-Sayed videos urged getting rid of the 4th of July tradition, raised replacing 2nd Amendment - What to watch in Tuesday’s primaries in Wisconsin, Minnesota, South Carolina and more - • Streaming NowStreaming NowStreaming Now MyPillow founder.',
 'Election denier.',
 'Former crack addict.',
 'Governor?',
 '- • For SubscribersFor Subscribers Have questions about today’s primaries?',
 'Ask CNN’s John King - He flew out of Turkey on a smaller craft; reporters and staffers were unknowingly on a decoy plane - • Video 0:22Video 0:22Video Rubio ignores question on if administration put Americans at risk on Air Force One 0:22 - • AnalysisAnalysis Trump’s secret plane swap is more problematic than it might seem Data centers - • For Subscribers• For SubscribersFor Subscribers ‘Brand-new everything’: How data centers transformed this small town’s economy - • Streaming NowStreaming NowStreaming Now The rise of AI brin

In [18]:
data['token_count']

769

In [19]:
data['sentence_count']

11

In [20]:
data['lexical_diversity']

0.5838751625487646

In [21]:
data['entropy']

8.039991890439019

# Put the Data in a DataFrame

In [22]:
crawl_df = pd.DataFrame([data])
crawl_df

,text,comments,page_title,links,images,domain,linked_domains,url,tokens,sentences,token_count,sentence_count,lexical_diversity,entropy
0,- • For Subscribers• For SubscribersFor Subscr...,,"Breaking News, Latest News and Videos | CNN",[https://arabic.cnn.com?hpt=header_edition-pic...,[https://cnn.com/media/sites/cnn/app-store-cnn...,cnn.com,"[arabic.cnn.com, bleacherreport.com, careers.w...",https://cnn.com,"[-, •, For, Subscribers•, For, SubscribersFor,...",[- • For Subscribers• For SubscribersFor Subsc...,769,11,0.583875,8.039992
